## Credit Card Customer Churn Prediction with Keras ANN
#### Practice binary classification notebook using a TensorFlow/Keras Artificial Neural Network to predict customer churn (Exited) from banking customer attributes.

In [1]:
import os
import kagglehub
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from tensorflow.keras import Sequential, Input
from tensorflow.keras.layers import Dense

## 1. Dataset Download & Initial Exploration
#### Download the rjmanoj/credit-card-customer-churn-prediction dataset via kagglehub, load Churn_Modelling.csv, and inspect shape, column types, missing values, and duplicates.

In [2]:
dataset_path = kagglehub.dataset_download("rjmanoj/credit-card-customer-churn-prediction")
csv_path = os.path.join(dataset_path, "Churn_Modelling.csv")

df = pd.read_csv(csv_path)
df.head()

Using Colab cache for faster access to the 'credit-card-customer-churn-prediction' dataset.


,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RowNumber        10000 non-null  int64  
 1   CustomerId       10000 non-null  int64  
 2   Surname          10000 non-null  object 
 3   CreditScore      10000 non-null  int64  
 4   Geography        10000 non-null  object 
 5   Gender           10000 non-null  object 
 6   Age              10000 non-null  int64  
 7   Tenure           10000 non-null  int64  
 8   Balance          10000 non-null  float64
 9   NumOfProducts    10000 non-null  int64  
 10  HasCrCard        10000 non-null  int64  
 11  IsActiveMember   10000 non-null  int64  
 12  EstimatedSalary  10000 non-null  float64
 13  Exited           10000 non-null  int64  
dtypes: float64(2), int64(9), object(3)
memory usage: 1.1+ MB


In [4]:
df.isnull().sum()

,0
RowNumber,0
CustomerId,0
Surname,0
CreditScore,0
Geography,0
Gender,0
Age,0
Tenure,0
Balance,0
NumOfProducts,0


In [5]:
df.duplicated().sum()

np.int64(0)

## 2. Preprocessing, Encoding & Stratified Split
#### Drop non-predictive identifier columns (RowNumber, CustomerId, Surname), one-hot encode Geography and Gender with drop_first=True, split 80/20 with stratification on Exited (random_state=42), and standardize features with StandardScaler.

In [6]:
df.drop(columns=["RowNumber", "CustomerId", "Surname"], inplace=True)
df = pd.get_dummies(df, columns=["Geography", "Gender"], drop_first=True)

X = df.drop(columns=["Exited"])
y = df["Exited"]

Train_X, Test_X, Train_y, Test_y = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
Train_X_scaled = scaler.fit_transform(Train_X)
Test_X_scaled = scaler.transform(Test_X)

## 3. Model Architecture, Compilation & Training
#### Build a Sequential neural network with one hidden layer (Dense(3, activation="sigmoid")) and a sigmoid output layer (Dense(1, activation="sigmoid")), compile with binary_crossentropy and adam, and train for 10 epochs.

In [7]:
model = Sequential([
    Input(shape=(Train_X_scaled.shape[1],)),
    Dense(3, activation="sigmoid"),
    Dense(1, activation="sigmoid")
])

model.summary()

model.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])
model.fit(Train_X_scaled, Train_y, epochs=10)

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 3)              │            36 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │             4 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 40 (160.00 B)

 Trainable params: 40 (160.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.2730 - loss: 0.7912
Epoch 2/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7469 - loss: 0.6311
Epoch 3/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7962 - loss: 0.5472
Epoch 4/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7962 - loss: 0.5040
Epoch 5/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7962 - loss: 0.4814
Epoch 6/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7962 - loss: 0.4674
Epoch 7/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7962 - loss: 0.4572
Epoch 8/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7962 - loss: 0.4492
Epoch 9/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7962 - loss: 0.4430
Epoch 10/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7962 - loss: 0.4381


## 4. Inspect Learned Weights & Evaluate on Test Set
#### Inspect the learned weights and biases of the hidden and output layers, threshold test probability predictions at 0.5, and report test accuracy.

In [9]:
model.layers[0].get_weights()

[array([[ 0.14623854,  0.11063677, -0.12238053],
        [-0.65816975, -1.6827644 ,  1.5516988 ],
        [ 0.1899837 , -0.0555255 , -0.10861944],
        [-0.2514335 , -0.45116183,  0.10636882],
        [ 0.69929135, -0.07455313, -0.34019223],
        [ 0.27462524,  0.09984614,  0.19075875],
        [ 0.6307462 ,  0.86013764, -0.48479545],
        [ 0.23010431, -0.15068631,  0.1392662 ],
        [-0.38186583, -0.67462933,  0.56838703],
        [ 0.05746708,  0.1055596 , -0.04398203],
        [ 0.16427827,  0.41127342, -0.6772214 ]], dtype=float32),
 array([ 0.12037569,  0.953902  , -0.5239528 ], dtype=float32)]

In [10]:
model.layers[1].get_weights()

[array([[-0.43796012],
        [-1.2021893 ],
        [ 0.6560515 ]], dtype=float32),
 array([-0.7547801], dtype=float32)]

In [11]:
pred = model.predict(Test_X_scaled)
y_pred = np.where(pred > 0.5, 1, 0)

print("Accuracy:", accuracy_score(Test_y, y_pred))

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
Accuracy: 0.7965
